In [1]:
from utils import logger
from pipeline.pdf_parser import GrobidPDFParser
from pipeline.sentence_extractor import extract_sentences
from pipeline.reference_resolver import ReferenceResolver
from pipeline.classifier import GeminiClassifier


In [2]:
logger.info("Starting the application...")

logger.info("Parsing the PDF and extracting information...")

parser = GrobidPDFParser(pdf_path="../papers/BERT.pdf")
parsed_paper = parser.parse()

logger.info("Successfully parsed the PDF. Extracted information:")
logger.info(f"Title: {parsed_paper.title}")
logger.info(f"Abstract: {parsed_paper.abstract}")

2026-05-09 14:43:48,576 - missing_citations - INFO - Starting the application...
2026-05-09 14:43:48,577 - missing_citations - INFO - Parsing the PDF and extracting information...
2026-05-09 14:44:03,240 - missing_citations - INFO - Successfully parsed the PDF. Extracted information:
2026-05-09 14:44:03,242 - missing_citations - INFO - Title: BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding
2026-05-09 14:44:03,243 - missing_citations - INFO - Abstract: We introduce a new language representation model called BERT, which stands for Bidirectional Encoder Representations from Transformers. Unlike recent language representation models (Peters et al., 2018a; BERT is conceptually simple and empirically powerful. It obtains new state-of-the-art results on eleven natural language processing tasks, including pushing the GLUE score to 80.5% (7.7% point absolute improvement), MultiNLI accuracy to 86.7% (4.6% absolute improvement), SQuAD v1.1 question answering Test 

In [3]:
logger.info("Extracting sentences from the parsed paper...")
sentences = extract_sentences(parsed_paper)
logger.info(f"Extracted {len(sentences)} sentences from the paper.")


2026-05-09 14:44:03,256 - missing_citations - INFO - Extracting sentences from the parsed paper...
2026-05-09 14:44:06,358 - missing_citations - INFO - Extracted 288 sentences from the paper.


In [4]:
logger.info("Resolving references in the paper...")
resolver = ReferenceResolver()
resolved_references = []

for ref in parsed_paper.references:
    resolved = resolver.resolve(ref)
    resolved_references.append(resolved)

logger.info("Resolved references:")
for ref, resolved in zip(parsed_paper.references, resolved_references):
    logger.info(f"Original: {ref}")
    logger.info(f"Resolved: {resolved}")
    print("---")

logger.info(f"Stats: {resolver.stats}")


2026-05-09 14:44:06,377 - missing_citations - INFO - Resolving references in the paper...
2026-05-09 14:44:53,606 - missing_citations - INFO - Resolved references:
2026-05-09 14:44:53,607 - missing_citations - INFO - Original: Alan Akbik, Duncan Blythe, and Roland Vollgraf. 2018. Contextual string embeddings for sequence labeling. In Proceedings of the 27th International Conference on Computational Linguistics, pages 1638-1649.
2026-05-09 14:44:53,608 - missing_citations - INFO - Resolved: ResolvedReference(raw_reference='Alan Akbik, Duncan Blythe, and Roland Vollgraf. 2018. Contextual string embeddings for sequence labeling. In Proceedings of the 27th International Conference on Computational Linguistics, pages 1638-1649.', resolved_paper_id='W2880875857', openalex_id='W2880875857', title='Contextual String Embeddings for Sequence Labeling', doi=None, method='openalex', confidence=1.0, unresolved_reason=None)
---
2026-05-09 14:44:53,609 - missing_citations - INFO - Original: Rami Al-R

In [5]:
from sentence_transformers import SentenceTransformer
from fastembed import SparseTextEmbedding
from database.qdrant import create_qdrant_client
from utils.config import config

logger.info("Initializing models and Qdrant client...")

# Initialize Qdrant client
qdrant_client = create_qdrant_client(config.QDRANT_URL)
logger.info(f"Connected to Qdrant at {config.QDRANT_URL}")

# Load dense model
logger.info(f"Loading dense model: {config.DENSE_MODEL}...")
dense_model = SentenceTransformer(config.DENSE_MODEL)
logger.info("Dense model loaded")

# Load sparse model
logger.info(f"Loading sparse model: {config.SPARSE_MODEL}...")
sparse_model = SparseTextEmbedding(model_name=config.SPARSE_MODEL)
logger.info("Sparse model loaded")

c:\Users\sampe\OneDrive\Desktop\facultate\licenta\missing-citations-identifier\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-05-09 14:45:11,240 - missing_citations - INFO - Initializing models and Qdrant client...
2026-05-09 14:45:11,632 - missing_citations - INFO - Connected to Qdrant at http://localhost:6333
2026-05-09 14:45:11,633 - missing_citations - INFO - Loading dense model: intfloat/multilingual-e5-large-instruct...


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 2394.76it/s]


2026-05-09 14:45:22,907 - missing_citations - INFO - Dense model loaded
2026-05-09 14:45:22,911 - missing_citations - INFO - Loading sparse model: prithivida/Splade_PP_en_v1...
2026-05-09 14:45:24,505 - missing_citations - INFO - Sparse model loaded


In [6]:
from ingest_missing import ingest_openalex_papers
from indexer import EmbeddingIndex

# 1. Collect the IDs of all papers that were found externally
missing_ids = [
    ref.openalex_id 
    for ref in resolved_references 
    if ref.method == "openalex_external" and ref.openalex_id
]

if missing_ids:
    # 2. You will need to pass your initialized EmbeddingIndex. 
    # (Assuming you already have your qdrant_client, dense_model, etc. initialized)
    embedding_idx = EmbeddingIndex(
        qdrant_client=qdrant_client,
        dense_model=dense_model,
        sparse_model=sparse_model
    )
    
    # 3. Fetch, insert to Postgres, embed, and insert to Qdrant!
    inserted_count = ingest_openalex_papers(missing_ids, embedding_idx)
    print(f"Successfully ingested {inserted_count} missing papers into the local corpus.")


In [7]:
"""
STAGE 4A - Citation Worthiness Classification with GEMINI CLASSIFIER
"""

logger.info("Classifying sentences for citation worthiness using Gemini Classifier...")
gemini_classifier = GeminiClassifier(model=config.CLASSIFIER_BACKUP[0], batch_size=31)

classified_sentences = gemini_classifier.classify_sentences(sentences[:90], parsed_paper.title, parsed_paper.abstract)

logger.info("Classification results:")
for i, sentence in enumerate(classified_sentences):
    logger.info(f"Classification {i}: {sentence.__dict__}")
    print("---")

2026-05-09 14:45:24,550 - missing_citations - INFO - Classifying sentences for citation worthiness using Gemini Classifier...
2026-05-09 14:45:25,739 - missing_citations - INFO - Sending batch 1/3 to Gemini...
2026-05-09 14:45:37,191 - missing_citations - INFO - Waiting 30 seconds before next Gemini API call...
2026-05-09 14:46:07,193 - missing_citations - INFO - Sending batch 2/3 to Gemini...
2026-05-09 14:46:14,081 - missing_citations - INFO - Waiting 30 seconds before next Gemini API call...
2026-05-09 14:46:44,083 - missing_citations - INFO - Sending batch 3/3 to Gemini...
2026-05-09 14:46:48,745 - missing_citations - INFO - Classification results:
2026-05-09 14:46:48,747 - missing_citations - INFO - Classification 0: {'text': 'We introduce a new language representation model called BERT, which stands for Bidirectional Encoder Representations from Transformers.', 'section': 'Abstract', 'position_in_section': 0.0, 'has_citation': False, 'citation_intent': <CitationIntent.METHOD: 'ME